In [1]:
# Ноутбук строит полный путь данных от сырья к аналитической витрине Bronze > Silver > Gold

# Слои(namespace):
# Bronze — сырые данные с намеренными ошибками (дубли, null, невалидные статусы)
# Silver — очищенные, типизированные, дедуплицированные данные
# Gold   — агрегированные витрины для аналитики

# Каталог:
# lakehouse (1 каталог, 3 namespace)

# Таблицы:
# lakehouse.bronze:
# Поля (даты, статусы, флаги) хранятся как STRING для того, чтобы  Bronze принимал сырое без преждевременной валидации.
# - raw_categories — Категории товаров: Electronics, Clothing, Books, Home, Sports. Без грязи, справочник из 5 строк.
# - raw_products — Товары: 100 товаров из products_seed.py. Грязь: is_active как STRING со значениями ["true", "false", None]: 86 активных, 8 неактивных, 6 null.
# - raw_customers — Клиенты: 500 уникальных + 15 полных дублей (около 3%). Грязь: created_at как STRING, дубли полностью повторяют строки.
# - raw_orders — Заказы: 5000 заказов за последние 12 месяцев. Грязь: статусы "DONE" и "shipped" как legacy/merge-мусор (по 47 строк, около 2%), order_date как STRING.
# - raw_order_items — Позиции заказов: 15037 строк, 2-4 позиции на заказ. Без грязи

# lakehouse.silver:
#  Типы корректные: DATE вместо STRING, category_name денормализуется в products, line_total вычисляется в order_items.
#  Silver чистит Bronze без изменения бизнес-смысла.
# - customers — Клиенты. Удаляем дубли, created_at приводится к DATE: 515 > 500.
# - products — Активные товары с названием категории. JOIN с raw_categories добавляет category_name, WHERE is_active = 'true' удаляет false/null: 100 > 86.
# - orders — Заказы с валидными статусами. WHERE status IN ('completed', 'pending', 'cancelled') удаляет DONE/shipped: 5000 > 4906. order_date приводится к DATE.
# - order_items — Позиции с line_total = quantity * unit_price как DECIMAL(12,2).

# lakehouse.gold:
#  Витрины под аналитические OLAP-запросы. Обе строятся только по orders.status = 'completed' —
#  pending и cancelled в выручку и RFM не идут.
# - mart_sales_by_category — Выручка по 5 категориям и месяцам (yyyy-MM).
# - mart_top_customers — RFM-витрина

In [3]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType,
    DecimalType, DateType, LongType
)
from decimal import Decimal

# Читаем credentials из env — не хардкодим в ноутбуке
client_id = os.environ["IVAN_CLIENT_ID"]
client_secret = os.environ["IVAN_CLIENT_SECRET"]
credential = f"{client_id}:{client_secret}"

# credential передаётся в builder — Iceberg читает его при initialize() каталога
# spark.conf.set() после getOrCreate() не попадает в properties каталога
spark = SparkSession.builder \
    .appName("lakehouse-bronze-silver-gold") \
    .config("spark.sql.catalog.lakehouse.credential", credential) \
    .getOrCreate()

spark

In [3]:
spark.sql("SHOW CATALOGS").show(truncate=False)

+-------------+
|catalog      |
+-------------+
|lakehouse    |
|spark_catalog|
+-------------+



In [4]:
# Namespaces уже созданы config_rbac_one_catalog.py
# IF NOT EXISTS — безопасно при повторном запуске
spark.sql("CREATE NAMESPACE IF NOT EXISTS lakehouse.bronze")
spark.sql("CREATE NAMESPACE IF NOT EXISTS lakehouse.silver")
spark.sql("CREATE NAMESPACE IF NOT EXISTS lakehouse.gold")
print("Namespaces ready")

Namespaces ready


In [5]:
# Bronze — Категории
spark.sql("""
    CREATE TABLE IF NOT EXISTS lakehouse.bronze.raw_categories (
        id          INT,
        name        STRING,
        description STRING
    ) USING iceberg
""")

# Bronze — Товары
spark.sql("""
    CREATE TABLE IF NOT EXISTS lakehouse.bronze.raw_products (
        id          INT,
        category_id INT,
        name        STRING,
        price       DECIMAL(10,2),
        is_active   STRING
    ) USING iceberg
""")

# Bronze — Клиенты
spark.sql("""
    CREATE TABLE IF NOT EXISTS lakehouse.bronze.raw_customers (
        id         INT,
        name       STRING,
        email      STRING,
        city       STRING,
        created_at STRING
    ) USING iceberg
""")

# Bronze — Заказы
spark.sql("""
    CREATE TABLE IF NOT EXISTS lakehouse.bronze.raw_orders (
        id           INT,
        customer_id  INT,
        status       STRING,
        total_amount DECIMAL(12,2),
        order_date   STRING
    ) USING iceberg
""")

# Bronze — Позиции заказов
spark.sql("""
    CREATE TABLE IF NOT EXISTS lakehouse.bronze.raw_order_items (
        id         INT,
        order_id   INT,
        product_id INT,
        quantity   INT,
        unit_price DECIMAL(10,2)
    ) USING iceberg
""")

print("Bronze tables created")

Bronze tables created


In [6]:
# Silver — Клиенты
spark.sql("""
    CREATE TABLE IF NOT EXISTS lakehouse.silver.customers (
        id         INT,
        name       STRING,
        email      STRING,
        city       STRING,
        created_at DATE
    ) USING iceberg
""")

# Silver — Товары
spark.sql("""
    CREATE TABLE IF NOT EXISTS lakehouse.silver.products (
        id            INT,
        category_id   INT,
        category_name STRING,
        name          STRING,
        price         DECIMAL(10,2)
    ) USING iceberg
""")

# Silver — Заказы
spark.sql("""
    CREATE TABLE IF NOT EXISTS lakehouse.silver.orders (
        id           INT,
        customer_id  INT,
        status       STRING,
        total_amount DECIMAL(12,2),
        order_date   DATE
    ) USING iceberg
""")

# Silver — Позиции заказов
spark.sql("""
    CREATE TABLE IF NOT EXISTS lakehouse.silver.order_items (
        id         INT,
        order_id   INT,
        product_id INT,
        quantity   INT,
        unit_price DECIMAL(10,2),
        line_total DECIMAL(12,2)
    ) USING iceberg
""")

print("Silver tables created")

Silver tables created


In [7]:
# Gold — Выручка по категориям
spark.sql("""
    CREATE TABLE IF NOT EXISTS lakehouse.gold.mart_sales_by_category (
        category_name STRING,
        month         STRING,
        total_revenue DECIMAL(14,2),
        order_count   BIGINT,
        avg_check     DECIMAL(12,2)
    ) USING iceberg
""")

# Gold — Топ клиенты
spark.sql("""
    CREATE TABLE IF NOT EXISTS lakehouse.gold.mart_top_customers (
        customer_id  INT,
        name         STRING,
        recency_days INT,
        frequency    BIGINT,
        monetary     DECIMAL(14,2),
        segment      STRING
    ) USING iceberg
""")

print("Gold tables created")

Gold tables created


In [8]:
print("Bronze tables")
spark.sql("SHOW TABLES IN lakehouse.bronze").show(truncate=False)

print("Silver tables")
spark.sql("SHOW TABLES IN lakehouse.silver").show(truncate=False)

print("Gold tables")
spark.sql("SHOW TABLES IN lakehouse.gold").show(truncate=False)

Bronze tables
+---------+---------------+-----------+
|namespace|tableName      |isTemporary|
+---------+---------------+-----------+
|bronze   |raw_products   |false      |
|bronze   |raw_customers  |false      |
|bronze   |raw_orders     |false      |
|bronze   |raw_categories |false      |
|bronze   |raw_order_items|false      |
+---------+---------------+-----------+

Silver tables
+---------+-----------+-----------+
|namespace|tableName  |isTemporary|
+---------+-----------+-----------+
|silver   |customers  |false      |
|silver   |products   |false      |
|silver   |orders     |false      |
|silver   |order_items|false      |
+---------+-----------+-----------+

Gold tables
+---------+----------------------+-----------+
|namespace|tableName             |isTemporary|
+---------+----------------------+-----------+
|gold     |mart_sales_by_category|false      |
|gold     |mart_top_customers    |false      |
+---------+----------------------+-----------+



In [9]:
# Загрузить seed-данные в Bronze
# Читаем готовые *_seed.py файлы и загружаем их в Bronze.
# Намеренно добавляем грязные данные для демонстрации роли Silver:
# - 3% дублей в raw_customers
# - raw_products.is_active: 86 true, 8 false, 6 null
# - 2% невалидных статусов в raw_orders
# - order_date и created_at хранятся как STRING

In [10]:
# Категории — читаем из categories_seed.py (5 фиксированных категорий)
from categories_seed import CATEGORIES_CATALOG

categories_data = [(c["id"], c["name"], c["description"]) for c in CATEGORIES_CATALOG]

categories_schema = StructType([
    StructField("id",          IntegerType(), False),
    StructField("name",        StringType(),  False),
    StructField("description", StringType(),  True),
])

df_categories = spark.createDataFrame(categories_data, schema=categories_schema)
df_categories.write.mode("append").saveAsTable("lakehouse.bronze.raw_categories")
print(f"raw_categories: {df_categories.count()} rows loaded")

raw_categories: 5 rows loaded


In [11]:
# Товары — читаем из products_seed.py (100 товаров с is_active детерминированно)
from products_seed import PRODUCTS_CATALOG

products_data = [
    (i, item["category_id"], item["name"], Decimal(str(item["price"])), item["is_active"])
    for i, item in enumerate(PRODUCTS_CATALOG, start=1)
]

products_schema = StructType([
    StructField("id",          IntegerType(),     False),
    StructField("category_id", IntegerType(),     False),
    StructField("name",        StringType(),      False),
    StructField("price",       DecimalType(10,2), False),
    StructField("is_active",   StringType(),      True),
])

df_products = spark.createDataFrame(products_data, schema=products_schema)
df_products.write.mode("append").saveAsTable("lakehouse.bronze.raw_products")
print(f"raw_products: {df_products.count()} rows loaded")
print("is_active distribution:")
df_products.groupBy("is_active").count().show(truncate=False)

raw_products: 100 rows loaded
is_active distribution:
+---------+-----+
|is_active|count|
+---------+-----+
|true     |86   |
|false    |8    |
|NULL     |6    |
+---------+-----+



In [12]:
# Клиенты — читаем из customers_seed.py (готовый seed с дублями)
from customers_seed import CUSTOMERS_CATALOG

customers_data = [
    (c["id"], c["name"], c["email"], c["city"], c["created_at"])
    for c in CUSTOMERS_CATALOG
]

customers_schema = StructType([
    StructField("id",         IntegerType(), False),
    StructField("name",       StringType(),  False),
    StructField("email",      StringType(),  False),
    StructField("city",       StringType(),  True),
    StructField("created_at", StringType(),  True),
])

df_customers = spark.createDataFrame(customers_data, schema=customers_schema)
df_customers.write.mode("append").saveAsTable("lakehouse.bronze.raw_customers")
raw_customers_count = df_customers.count()
raw_customers_distinct_count = df_customers.dropDuplicates().count()
print(f"raw_customers: {raw_customers_count} rows loaded, duplicates={raw_customers_count - raw_customers_distinct_count}")

raw_customers: 515 rows loaded, duplicates=15


In [13]:
# Заказы — читаем из orders_seed.py (total_amount = SUM(quantity * unit_price))
from orders_seed import ORDERS_CATALOG

orders_data = [
    (o["id"], o["customer_id"], o["status"], Decimal(str(o["total_amount"])), o["order_date"])
    for o in ORDERS_CATALOG
]

orders_schema = StructType([
    StructField("id",           IntegerType(),     False),
    StructField("customer_id",  IntegerType(),     False),
    StructField("status",       StringType(),      False),
    StructField("total_amount", DecimalType(12,2), False),
    StructField("order_date",   StringType(),      False),
])

df_orders = spark.createDataFrame(orders_data, schema=orders_schema)
df_orders.write.mode("append").saveAsTable("lakehouse.bronze.raw_orders")
print(f"raw_orders: {df_orders.count()} rows loaded")
print("Status distribution:")
df_orders.groupBy("status").count().orderBy(F.col("count").desc()).show(truncate=False)

raw_orders: 5000 rows loaded
Status distribution:
+---------+-----+
|status   |count|
+---------+-----+
|completed|3521 |
|pending  |997  |
|cancelled|388  |
|shipped  |47   |
|DONE     |47   |
+---------+-----+



In [14]:
# Позиции — читаем из order_items_seed.py (unit_price рассчитан от products.price)
from order_items_seed import ORDER_ITEMS_CATALOG

order_items_data = [
    (it["id"], it["order_id"], it["product_id"], it["quantity"], Decimal(str(it["unit_price"])))
    for it in ORDER_ITEMS_CATALOG
]

order_items_schema = StructType([
    StructField("id",         IntegerType(),     False),
    StructField("order_id",   IntegerType(),     False),
    StructField("product_id", IntegerType(),     False),
    StructField("quantity",   IntegerType(),     False),
    StructField("unit_price", DecimalType(10,2), False),
])

df_order_items = spark.createDataFrame(order_items_data, schema=order_items_schema)
df_order_items.write.mode("append").saveAsTable("lakehouse.bronze.raw_order_items")
print(f"raw_order_items: {df_order_items.count()} rows loaded")

raw_order_items: 15037 rows loaded


In [15]:
print("Bronze row counts")
for table in ["raw_categories", "raw_products", "raw_customers", "raw_orders", "raw_order_items"]:
    cnt = spark.sql(f"""
        SELECT
            COUNT(*) AS cnt
        FROM lakehouse.bronze.{table}
    """).collect()[0]["cnt"]
    print(f"  lakehouse.bronze.{table}: {cnt}")

Bronze row counts
  lakehouse.bronze.raw_categories: 5
  lakehouse.bronze.raw_products: 100
  lakehouse.bronze.raw_customers: 515
  lakehouse.bronze.raw_orders: 5000
  lakehouse.bronze.raw_order_items: 15037


In [16]:
# Трансформация: Bronze > Silver
# Очищаем и типизируем данные из Bronze

In [17]:
spark.sql("""
    INSERT INTO lakehouse.silver.customers
    SELECT DISTINCT
        id,
        name,
        email,
        city,
        CAST(created_at AS DATE) AS created_at
    FROM lakehouse.bronze.raw_customers
""")

cnt = spark.sql("""
    SELECT
        COUNT(*) AS cnt
    FROM lakehouse.silver.customers
""").collect()[0]["cnt"]
bronze_cnt = spark.sql("""
    SELECT
        COUNT(*) AS cnt
    FROM lakehouse.bronze.raw_customers
""").collect()[0]["cnt"]
print(f"silver.customers: {cnt} rows (bronze={bronze_cnt}, filtered={bronze_cnt - cnt})")

silver.customers: 500 rows (bronze=515, filtered=15)


In [18]:
spark.sql("""
    INSERT INTO lakehouse.silver.products
    SELECT
        p.id,
        p.category_id,
        c.name AS category_name,
        p.name,
        p.price
    FROM lakehouse.bronze.raw_products p
    JOIN lakehouse.bronze.raw_categories c
      ON p.category_id = c.id
    WHERE p.is_active = 'true'
""")

cnt = spark.sql("""
    SELECT
        COUNT(*) AS cnt
    FROM lakehouse.silver.products
""").collect()[0]["cnt"]
bronze_cnt = spark.sql("""
    SELECT
        COUNT(*) AS cnt
    FROM lakehouse.bronze.raw_products
""").collect()[0]["cnt"]
print(f"silver.products: {cnt} rows (bronze={bronze_cnt}, filtered={bronze_cnt - cnt})")

spark.sql("""
    SELECT
        category_name,
        COUNT(*) AS cnt
    FROM lakehouse.silver.products
    GROUP BY category_name
""").show(truncate=False)

silver.products: 86 rows (bronze=100, filtered=14)
+-------------+---+
|category_name|cnt|
+-------------+---+
|Books        |19 |
|Home         |16 |
|Electronics  |19 |
|Clothing     |17 |
|Sports       |15 |
+-------------+---+



In [19]:
spark.sql("""
    INSERT INTO lakehouse.silver.orders
    SELECT
        id,
        customer_id,
        status,
        total_amount,
        CAST(order_date AS DATE) AS order_date
    FROM lakehouse.bronze.raw_orders
    WHERE status IN ('completed', 'pending', 'cancelled')
""")

cnt = spark.sql("""
    SELECT
        COUNT(*) AS cnt
    FROM lakehouse.silver.orders
""").collect()[0]["cnt"]
bronze_cnt = spark.sql("""
    SELECT
        COUNT(*) AS cnt
    FROM lakehouse.bronze.raw_orders
""").collect()[0]["cnt"]
print(f"silver.orders: {cnt} rows (bronze={bronze_cnt}, filtered={bronze_cnt - cnt})")

spark.sql("""
    SELECT
        status,
        COUNT(*) AS cnt
    FROM lakehouse.silver.orders
    GROUP BY status
""").show(truncate=False)

silver.orders: 4906 rows (bronze=5000, filtered=94)
+---------+----+
|status   |cnt |
+---------+----+
|completed|3521|
|cancelled|388 |
|pending  |997 |
+---------+----+



In [20]:
spark.sql("""
    INSERT INTO lakehouse.silver.order_items
    SELECT
        id,
        order_id,
        product_id,
        quantity,
        unit_price,
        CAST(quantity * unit_price AS DECIMAL(12,2)) AS line_total
    FROM lakehouse.bronze.raw_order_items
    WHERE quantity > 0 AND unit_price > 0
""")

cnt = spark.sql("""
    SELECT
        COUNT(*) AS cnt
    FROM lakehouse.silver.order_items
""").collect()[0]["cnt"]
print(f"silver.order_items: {cnt} rows")

spark.sql("""
    SELECT
        MIN(line_total),
        MAX(line_total),
        AVG(line_total)
    FROM lakehouse.silver.order_items
""").show(truncate=False)

silver.order_items: 15037 rows
+---------------+---------------+---------------+
|min(line_total)|max(line_total)|avg(line_total)|
+---------------+---------------+---------------+
|8.53           |9486.65        |830.053864     |
+---------------+---------------+---------------+



In [21]:
print("Bronze vs Silver row counts")
layers = [
    ("lakehouse.bronze.raw_customers",   "lakehouse.silver.customers"),
    ("lakehouse.bronze.raw_products",    "lakehouse.silver.products"),
    ("lakehouse.bronze.raw_orders",      "lakehouse.silver.orders"),
    ("lakehouse.bronze.raw_order_items", "lakehouse.silver.order_items"),
]
for bronze_t, silver_t in layers:
    b = spark.sql(f"""
        SELECT
            COUNT(*) AS cnt
        FROM {bronze_t}
    """).collect()[0]["cnt"]
    s = spark.sql(f"""
        SELECT
            COUNT(*) AS cnt
        FROM {silver_t}
    """).collect()[0]["cnt"]
    print(f"  {bronze_t.split('.')[-1]}: bronze={b} > silver={s} (удалено {b-s})")

Bronze vs Silver row counts
  raw_customers: bronze=515 > silver=500 (удалено 15)
  raw_products: bronze=100 > silver=86 (удалено 14)
  raw_orders: bronze=5000 > silver=4906 (удалено 94)
  raw_order_items: bronze=15037 > silver=15037 (удалено 0)


In [22]:
# Агрегация: Silver > Gold
# Строим две витрины:
# - mart_sales_by_category — выручка по категориям товаров и месяцам
# - mart_top_customers     — RFM-сегментация покупателей (High/Mid/Low)

In [23]:
spark.sql("""
    INSERT INTO lakehouse.gold.mart_sales_by_category
    SELECT
        p.category_name,
        DATE_FORMAT(o.order_date, 'yyyy-MM')                              AS month,
        CAST(SUM(oi.line_total) AS DECIMAL(14,2))                         AS total_revenue,
        COUNT(DISTINCT o.id)                                              AS order_count,
        CAST(SUM(oi.line_total) / COUNT(DISTINCT o.id) AS DECIMAL(12,2))  AS avg_check
    FROM lakehouse.silver.order_items oi
    JOIN lakehouse.silver.orders o ON oi.order_id = o.id
    JOIN lakehouse.silver.products p ON oi.product_id = p.id
    WHERE o.status = 'completed'
    GROUP BY p.category_name, DATE_FORMAT(o.order_date, 'yyyy-MM')
    ORDER BY month, category_name
""")

cnt = spark.sql("""
    SELECT
        COUNT(*) AS cnt
    FROM lakehouse.gold.mart_sales_by_category
""").collect()[0]["cnt"]
print(f"mart_sales_by_category: {cnt} rows")

spark.sql("""
    SELECT
        category_name,
        SUM(total_revenue)  AS total,
        SUM(order_count)    AS orders
    FROM lakehouse.gold.mart_sales_by_category
    GROUP BY category_name ORDER BY total DESC
""").show(truncate=False)

mart_sales_by_category: 65 rows
+-------------+----------+------+
|category_name|total     |orders|
+-------------+----------+------+
|Electronics  |5242567.02|1865  |
|Sports       |1552199.15|1511  |
|Home         |1094174.70|1568  |
|Clothing     |590408.07 |1665  |
|Books        |162499.96 |1833  |
+-------------+----------+------+



In [24]:
spark.sql("""
    INSERT INTO lakehouse.gold.mart_top_customers
    WITH rfm AS (
        SELECT
            c.id AS customer_id,
            c.name,
            DATEDIFF(CURRENT_DATE(), MAX(o.order_date)) AS recency_days,
            COUNT(DISTINCT o.id)                        AS frequency,
            CAST(SUM(oi.line_total)                     AS DECIMAL(14,2)) AS monetary
        FROM lakehouse.silver.customers c
        JOIN lakehouse.silver.orders o ON c.id = o.customer_id
        JOIN lakehouse.silver.order_items oi ON o.id = oi.order_id
        WHERE o.status = 'completed'
        GROUP BY c.id, c.name
    ),
    ranked AS (
        SELECT *,
            PERCENT_RANK() OVER (ORDER BY monetary DESC) AS pct_rank
        FROM rfm
    )
    SELECT
        customer_id,
        name,
        recency_days,
        frequency,
        monetary,
        CASE
            WHEN pct_rank <= 0.20 THEN 'High'
            WHEN pct_rank <= 0.50 THEN 'Mid'
            ELSE 'Low'
        END AS segment
    FROM ranked
""")

cnt = spark.sql("""
    SELECT
        COUNT(*) AS cnt
    FROM lakehouse.gold.mart_top_customers
""").collect()[0]["cnt"]
print(f"mart_top_customers: {cnt} customers с RFM-сегментацией")

spark.sql("""
    SELECT
        segment,
        COUNT(*)        AS cnt,
        AVG(monetary)   AS avg_monetary,
        AVG(frequency)  AS avg_orders
    FROM lakehouse.gold.mart_top_customers
    GROUP BY segment ORDER BY avg_monetary DESC
""").show(truncate=False)

mart_top_customers: 500 customers с RFM-сегментацией
+-------+---+------------+-----------------+
|segment|cnt|avg_monetary|avg_orders       |
+-------+---+------------+-----------------+
|High   |100|31907.722600|9.71             |
|Mid    |150|20197.928600|8.053333333333333|
|Low    |250|9685.549400 |5.368            |
+-------+---+------------+-----------------+



In [5]:
print("=" * 55)
print("ИТОГОВАЯ СВОДКА — COUNT(*) ПО ВСЕМ 11 ТАБЛИЦАМ")
print("=" * 55)

all_tables = [
    ("Bronze", "lakehouse.bronze.raw_categories"),
    ("Bronze", "lakehouse.bronze.raw_products"),
    ("Bronze", "lakehouse.bronze.raw_customers"),
    ("Bronze", "lakehouse.bronze.raw_orders"),
    ("Bronze", "lakehouse.bronze.raw_order_items"),
    ("Silver", "lakehouse.silver.customers"),
    ("Silver", "lakehouse.silver.products"),
    ("Silver", "lakehouse.silver.orders"),
    ("Silver", "lakehouse.silver.order_items"),
    ("Gold",   "lakehouse.gold.mart_sales_by_category"),
    ("Gold",   "lakehouse.gold.mart_top_customers"),
]

for layer, table in all_tables:
    cnt = spark.sql(f"""
        SELECT
            COUNT(*) AS cnt
        FROM {table}
    """).collect()[0]["cnt"]
    print(f"  [{layer:6s}] {table.split('.')[-1]:30s}: {cnt:>8,}")

ИТОГОВАЯ СВОДКА — COUNT(*) ПО ВСЕМ 11 ТАБЛИЦАМ
  [Bronze] raw_categories                :        5
  [Bronze] raw_products                  :      100
  [Bronze] raw_customers                 :      515
  [Bronze] raw_orders                    :    5,000
  [Bronze] raw_order_items               :   15,037
  [Silver] customers                     :      500
  [Silver] products                      :       86
  [Silver] orders                        :    4,906
  [Silver] order_items                   :   15,037
  [Gold  ] mart_sales_by_category        :       65
  [Gold  ] mart_top_customers            :      500
